# 🧪 Unity Catalog — Part 2: Hands-On Notebook

### Build it yourself

Welcome to **Part 2**! 👋

This notebook assumes you've already been through **Part 1 — Learning**, where we covered
*what* Unity Catalog is and *why* it exists. Here, we focus purely on **doing**.

**How this notebook works:**

- 📝 Each section gives you a **Task** describing what to build
- ✍️ You write the code yourself in the cell below it (look for `-- TODO` / `# TODO`)
- 💡 A **Solution** cell follows — try not to peek until you've had a go!

> ⚠️ You need `CREATE CATALOG` privileges on your Metastore to run most of the examples
> below. If you don't have that privilege, pair up with someone who does, or follow along by
> reading the solutions during the KT session.

Let's build the same Microsoft-style catalog structure we discussed in Part 1:

```
sales_catalog
 ├── gold
 │    ├── customers
 │    ├── customers_external
 │    └── contracts_volume (Volume)
 └── silver
```

## 🔎 Step 0 — Look Around First

**Task:** Before creating anything, check what catalogs already exist in your Metastore.

In [0]:
%sql
SHOW CATALOGS;

💡 **Solution**
```sql
SHOW CATALOGS;
```

## 1️⃣ Task — Create Your Catalog

**Task:** Create a catalog named `sales_catalog` with a comment describing that it holds
Microsoft Sales data. Use `IF NOT EXISTS` so it's safe to re-run.

In [0]:
%sql
-- TODO: write your CREATE CATALOG statement here

💡 **Solution**
```sql
CREATE CATALOG IF NOT EXISTS sales_catalog
COMMENT 'Catalog for all Microsoft Sales data';
```

## 2️⃣ Task — Switch to Your Catalog

**Task:** Set `sales_catalog` as the default catalog for this session, then confirm it
worked by checking the current catalog.

In [0]:
%sql
-- TODO: switch to sales_catalog, then check the current catalog

💡 **Solution**
```sql
USE CATALOG sales_catalog;
SELECT current_catalog();
```

## 3️⃣ Task — Create Two Schemas

**Task:** Inside `sales_catalog`, create:
- a `gold` schema — for curated, reporting-ready data
- a `silver` schema — for cleaned but not-yet-curated data

Add a meaningful comment to each.

In [0]:
%sql
-- TODO: create the gold schema

-- TODO: create the silver schema

💡 **Solution**
```sql
CREATE SCHEMA IF NOT EXISTS sales_catalog.gold
COMMENT 'Curated sales data, ready for reporting';

CREATE SCHEMA IF NOT EXISTS sales_catalog.silver
COMMENT 'Cleaned sales data';
```

## 4️⃣ Task — Create a Table and Load Data

**Task:** In `sales_catalog.gold`, create a table called `customers` with these columns:

| Column | Type |
|---|---|
| customer_id | INT |
| customer_name | STRING |
| region | STRING |
| total_purchase_usd | DOUBLE |

Then insert at least 3 rows of sample data.

In [0]:
%sql
-- TODO: CREATE TABLE sales_catalog.gold.customers ...


-- TODO: INSERT INTO sales_catalog.gold.customers VALUES ...

💡 **Solution**
```sql
CREATE TABLE IF NOT EXISTS sales_catalog.gold.customers (
  customer_id INT,
  customer_name STRING,
  region STRING,
  total_purchase_usd DOUBLE
)
COMMENT 'Customer master data for Sales';

INSERT INTO sales_catalog.gold.customers VALUES
  (1, 'Contoso Ltd', 'US', 125000.50),
  (2, 'Fabrikam Inc', 'EU', 87250.00),
  (3, 'Northwind Traders', 'APAC', 45230.75);
```

## 5️⃣ Task — Query and Explore the Table

**Task:** Write a query to view all rows in `sales_catalog.gold.customers`. Then, in a
separate cell, write the PySpark equivalent using `spark.sql(...)`.

In [0]:
%sql
-- TODO: SELECT * FROM sales_catalog.gold.customers;

In [0]:
# TODO: write the PySpark equivalent that displays the same result

💡 **Solution**
```sql
SELECT * FROM sales_catalog.gold.customers;
```
```python
spark.sql("SELECT * FROM sales_catalog.gold.customers").display()
```

## 6️⃣ Task — Inspect Metadata

**Task:** Run three separate commands to:
1. List all tables inside `sales_catalog.gold`
2. Show the column definitions of `customers`
3. Show the full storage/location details of `customers`

In [0]:
%sql
-- TODO: list tables in sales_catalog.gold

-- TODO: show column definitions for customers

-- TODO: show full details (including storage location) for customers

💡 **Solution**
```sql
SHOW TABLES IN sales_catalog.gold;
DESCRIBE TABLE sales_catalog.gold.customers;
DESCRIBE DETAIL sales_catalog.gold.customers;
```

**Check yourself:** in the `DESCRIBE DETAIL` output, find the `location` column — this proves
the data lives in cloud storage, not "inside" Unity Catalog.

## 7️⃣ Task — Session Context

**Task:** Write one query that returns your current catalog, current schema, and the current
metastore, all at once.

In [0]:
%sql
-- TODO: return current_catalog(), current_schema(), and current_metastore()

💡 **Solution**
```sql
SELECT current_catalog(), current_schema(), current_metastore();
```

## 8️⃣ Task — Explore with INFORMATION_SCHEMA

**Task:** Write three `INFORMATION_SCHEMA` queries against `sales_catalog` that answer:
1. What columns exist across every table (name, type, position)?
2. What privileges have been granted, and to whom?
3. Which tables are `EXTERNAL` (as opposed to managed)?

In [0]:
%sql
-- TODO: query INFORMATION_SCHEMA.COLUMNS

-- TODO: query INFORMATION_SCHEMA.TABLE_PRIVILEGES

-- TODO: query INFORMATION_SCHEMA.TABLES filtered to table_type = 'EXTERNAL'

💡 **Solution**
```sql
SELECT * FROM sales_catalog.INFORMATION_SCHEMA.COLUMNS;

SELECT * FROM sales_catalog.INFORMATION_SCHEMA.TABLE_PRIVILEGES;

SELECT * FROM sales_catalog.INFORMATION_SCHEMA.TABLES
WHERE table_type = 'EXTERNAL';
```

**Bonus:** write a query against `INFORMATION_SCHEMA.TABLES` that returns only tables owned
by you (hint: `current_user()`).

## 9️⃣ Task — Create a Volume and Add a File

**Task:**
1. Create a Volume called `contracts_volume` inside `sales_catalog.gold`
2. List its (empty) contents
3. Copy a file into it using `dbutils.fs.cp`, then list it again to confirm

In [0]:
%sql
-- TODO: CREATE VOLUME sales_catalog.gold.contracts_volume ...

-- TODO: LIST the (empty) volume

In [0]:
# TODO: use dbutils.fs.cp to copy a file into the volume, then list it again

💡 **Solution**
```sql
CREATE VOLUME IF NOT EXISTS sales_catalog.gold.contracts_volume
COMMENT 'Governed storage for Sales contract PDFs';

LIST '/Volumes/sales_catalog/gold/contracts_volume';
```
```python
dbutils.fs.cp(
    "file:/tmp/contoso_msa_2026.pdf",
    "/Volumes/sales_catalog/gold/contracts_volume/contoso_msa_2026.pdf"
)
display(dbutils.fs.ls("/Volumes/sales_catalog/gold/contracts_volume"))
```

**Your turn:** create a second volume, `product_images_volume`, for product photos.

## 🔟 Task — Storage Credential & External Location

> ⚠️ These usually require Metastore Admin privileges. If you don't have them, write the
> code anyway (for practice) and compare it against the solution — you don't have to run it.

**Task:**
1. Create a Storage Credential backed by an Azure Managed Identity (or an AWS IAM Role, if
   your org uses AWS)
2. Create an External Location that uses that credential to point at a cloud storage path
3. Create an **external** table on top of that location

In [0]:
%sql
-- TODO: CREATE STORAGE CREDENTIAL ...

-- TODO: CREATE EXTERNAL LOCATION ... WITH (STORAGE CREDENTIAL ...)

-- TODO: CREATE TABLE ... LOCATION '...'

💡 **Solution**
```sql
-- Azure example
CREATE STORAGE CREDENTIAL IF NOT EXISTS microsoft_sales_credential
AZURE_MANAGED_IDENTITY
  ACCESS_CONNECTOR_ID 'subscriptions/<sub-id>/resourceGroups/<rg-name>/providers/Microsoft.Databricks/accessConnectors/<connector-name>'
COMMENT 'Managed identity used to access Microsoft Sales data in ADLS Gen2';

-- AWS equivalent, for reference:
-- CREATE STORAGE CREDENTIAL IF NOT EXISTS microsoft_sales_credential
-- AWS_IAM_ROLE
--   ROLE_ARN 'arn:aws:iam::123456789012:role/unity-catalog-sales-role'
-- COMMENT 'IAM role used to access Microsoft Sales data in S3';

CREATE EXTERNAL LOCATION IF NOT EXISTS sales_external_location
URL 'abfss://sales@microsoftdatalake.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL microsoft_sales_credential)
COMMENT 'External location pointing to raw Sales data in ADLS';

CREATE TABLE IF NOT EXISTS sales_catalog.gold.customers_external (
  customer_id INT,
  customer_name STRING,
  region STRING
)
LOCATION 'abfss://sales@microsoftdatalake.dfs.core.windows.net/external/customers/';
```

**Check yourself:** run `SHOW EXTERNAL LOCATIONS;` and `DESCRIBE EXTERNAL LOCATION
sales_external_location;` — can you see which credential it's tied to?

## 1️⃣1️⃣ Task — Grant and Revoke Permissions

**Task:**
1. Grant `SELECT` on `sales_catalog.gold.customers` to a group called `finance_team`
2. Grant the `USE CATALOG` / `USE SCHEMA` privileges the team also needs to actually reach
   the table
3. Confirm the grants with `SHOW GRANTS`
4. Revoke the `SELECT` grant again

In [0]:
%sql
-- TODO: GRANT SELECT ...

-- TODO: GRANT USE CATALOG / USE SCHEMA ...

-- TODO: SHOW GRANTS ...

-- TODO: REVOKE SELECT ...

💡 **Solution**
```sql
GRANT SELECT ON TABLE sales_catalog.gold.customers TO `finance_team`;
GRANT USE CATALOG ON CATALOG sales_catalog TO `finance_team`;
GRANT USE SCHEMA ON SCHEMA sales_catalog.gold TO `finance_team`;

SHOW GRANTS ON TABLE sales_catalog.gold.customers;

REVOKE SELECT ON TABLE sales_catalog.gold.customers FROM `finance_team`;
```

**Your turn:** grant `SELECT` to a `marketing_team` group instead, and confirm it with
`SHOW GRANTS`.

## 1️⃣2️⃣ Task — Build a Row Filter

**Task:** Restrict `sales_catalog.gold.customers` so that members of `apac_sales_team` only
ever see rows where `region = 'APAC'` (metastore admins should still see everything).

1. Write a SQL function that implements this rule
2. Attach it to the table as a row filter
3. Write the query a filtered user would run — notice it needs **no changes**
4. Write the command to remove the filter again

In [0]:
%sql
-- TODO: CREATE OR REPLACE FUNCTION ... (the row filter rule)

-- TODO: ALTER TABLE ... SET ROW FILTER ...

-- TODO: SELECT * FROM sales_catalog.gold.customers;  (as a filtered user, mentally trace the output)

-- TODO: ALTER TABLE ... DROP ROW FILTER;

💡 **Solution**
```sql
CREATE OR REPLACE FUNCTION sales_catalog.gold.region_row_filter(region STRING)
RETURN
  is_account_group_member('apac_sales_team') AND region = 'APAC'
  OR is_account_group_member('metastore_admins');

ALTER TABLE sales_catalog.gold.customers
SET ROW FILTER sales_catalog.gold.region_row_filter ON (region);

SELECT * FROM sales_catalog.gold.customers;
-- A member of apac_sales_team now only ever sees region = 'APAC' rows.

ALTER TABLE sales_catalog.gold.customers DROP ROW FILTER;
```

## 1️⃣3️⃣ Task — Build a Column Mask

**Task:** Mask the `total_purchase_usd` column so only members of `finance_admins` can see
the real value — everyone else should see `NULL`.

1. Write a SQL function that implements this rule
2. Attach it to the column as a mask
3. Write the command to remove the mask again

In [0]:
%sql
-- TODO: CREATE OR REPLACE FUNCTION ... (the masking rule)

-- TODO: ALTER TABLE ... ALTER COLUMN ... SET MASK ...

-- TODO: ALTER TABLE ... ALTER COLUMN ... DROP MASK;

💡 **Solution**
```sql
CREATE OR REPLACE FUNCTION sales_catalog.gold.mask_purchase_amount(total_purchase_usd DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN is_account_group_member('finance_admins') THEN total_purchase_usd
    ELSE NULL
  END;

ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN total_purchase_usd
SET MASK sales_catalog.gold.mask_purchase_amount;

ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN total_purchase_usd DROP MASK;
```

**Your turn:** write a mask for `customer_name` that shows `'REDACTED'` to everyone except
`sales_admins`.

## 🏆 Capstone Challenge

Now put it all together — **without solutions this time**. Build this from scratch:

1. A new catalog: `hr_catalog`
2. A schema: `hr_catalog.gold`
3. A table: `hr_catalog.gold.employees` with columns `employee_id INT`, `employee_name
   STRING`, `department STRING`, `salary DOUBLE`
4. Insert 3 sample rows across at least 2 departments
5. A row filter so only `hr_admins` can see rows outside their own department
6. A column mask on `salary` so only `hr_admins` can see the real value
7. A `GRANT SELECT` to a group called `hr_readonly`

Write and run each step yourself in the cell below.

In [0]:
# 🏆 Your capstone solution goes here.
# Tip: work one numbered step at a time, and re-run SHOW GRANTS / DESCRIBE DETAIL
# after each step to confirm it did what you expected.

## 🎉 Congratulations!

If you've made it through every task, you've now:
- Built a full catalog → schema → table hierarchy from scratch
- Queried metadata with `INFORMATION_SCHEMA`
- Created a Volume and moved a file into governed storage
- Wired up a Storage Credential and External Location
- Granted and revoked permissions
- Written your own row filter and column mask
- Designed and secured an entirely new catalog on your own in the capstone

You're ready to work with Unity Catalog on a real project. 🚀